# hdrs

> App integration: the headers that carry the registry's assets

In [ ]:
#| default_exp hdrs

An app integrates answerui by passing its headers to `FastHTML` (or `fast_app`). However, the registry can change after that call, so the headers can't be built once at construction time; they have to be rendered fresh for each response.

In [ ]:
from fastcore.utils import *
from fasthtml.common import Style, Script, Link
from answerui.registry import Registry
import answerui.behaviors  # installs Registry.build_js
import hashlib
from starlette.responses import Response

In [ ]:
from fasthtml.common import *
from answerui.registry import reg, hypelt
from fastcore.test import *

When a FastHTML page is rendered, any object with an `__ft__` method is asked for its markup at that moment. We make use of this: `_Lazy` wraps a function to make it get called per response, and a registry's `hdrs` returns its stylesheet and behavior script that way. The headers belong to the registry, so a registry of one's own renders its own page with nothing else changed:

In [ ]:
class _Lazy:
    def __init__(self, f): self.f = f
    def __ft__(self): return self.f()

@patch
def hdrs(self:Registry):
    "Headers for `fast_app(hdrs=...)`: the stylesheet and behavior script, rendered from this registry per response"
    return _Lazy(lambda: Style(self.build_css())), _Lazy(lambda: Script(self.build_js()))


To see the deferral, build the headers first and define a component afterwards. Rendering the stylesheet header then includes it.

In [ ]:
hdrs = reg.hdrs()
LateCard = hypelt('LateCard', css='late-card { display: block; color: teal; }', layer='components',
    on={('&', 'click'): r"el.remove()"}, doc='Defined after the app; click to remove.')
css = to_xml(hdrs[0])
print(css[css.index('/* LateCard.base */'):][:60])

/* LateCard.base */
late-card { display: block; color: teal;


Redefining it changes the next render. There is no rebuild step and nothing to invalidate:

In [ ]:
LateCard = hypelt('LateCard', css='late-card { display: block; color: navy; }', layer='components', doc='Defined after the app.')
css = to_xml(hdrs[0])
assert 'navy' in css and 'teal' not in css

## A whole page

That is all an app needs. FastHTML's test client fetches a page without starting a server, and the response carries htmx, then our stylesheet and script, in the head.

In [ ]:
app = FastHTML(hdrs=reg.hdrs())
rt = app.route

@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

page = Client(app).get('/').text
print(re.sub(r'(?s)(<(style|script)>).*?(?=</\2>)', r'\1…', page))

<!doctype html>
<html>
  <head>
    <title>Hello</title>
    <link rel="canonical" href="https://testserver/">
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.13/fasthtml.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js"></script><script src="https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js"></script>    <style>…</style>
<script>…</script>  </head>
  <body>
<main class="container">      <h1>Hello</h1>
<late-card>Defined after the app.</late-card></main>  </body>
</html>



`fast_app` works the same way, but it adds Pico CSS by default, so we turn that off.

In [ ]:
app, rt = fast_app(hdrs=reg.hdrs(), pico=False)
@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

assert 'pico' not in (r:= Client(app).get('/').text)
assert 'late-card' in r

## Serving the assets

Inline headers carry the whole stylesheet and script in every response. Served from a URL instead, the browser fetches them once and keeps them, but a fixed URL would lose what the lazy headers give: a change to the registry reaching the next page. A name that carries a hash of the content keeps that property. `serve` mounts one route on the app that answers with the registry's current stylesheet or script, under any name, and marks the response immutable. The headers it adds stay lazy: each response builds the assets, hashes them, and links to a name that carries the hash. While the registry is unchanged every page links the same name and the browser has it cached. When a rule changes, the next page links a new name, which the browser has never seen, and fetches. An old name still answers with the current build, so a tab opened before the change is never left without a stylesheet. `serve` also returns the two headers, for an app that builds its own `head`.

In [ ]:
def _digest(s): return hashlib.sha256(s.encode()).hexdigest()[:12]

@patch
def serve(
    self:Registry,
    app,              # The FastHTML app to mount the route on and add the headers to
    path='/answerui', # URL prefix the assets are served under
):
    "Serve the stylesheet and script from `app` under content-hashed names, linked from every page; returns the two headers, for an app that builds its own head"
    assets = {'css': (self.build_css, 'text/css'), 'js': (self.build_js, 'text/javascript')}
    @app.get(f'{path}/{{name}}')
    def asset(name: str):
        build, mime = assets.get(name.rsplit('.', 1)[-1], (None, None))
        if not build: return Response(status_code=404)
        return Response(build(), media_type=mime, headers={'Cache-Control': 'public, max-age=31536000, immutable'})
    hdrs = (_Lazy(lambda: Link(rel='stylesheet', href=f'{path}/{_digest(self.build_css())}.css')),
        _Lazy(lambda: Script(src=f'{path}/{_digest(self.build_js())}.js')))
    app.hdrs = [*app.hdrs, *hdrs]
    return hdrs

In [ ]:
app = FastHTML()
rt = app.route
reg.serve(app)

@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

page = Client(app).get('/').text
href = re.search(r'href="(/answerui/[^"]+\.css)"', page)[1]
asset = Client(app).get(href)
test_eq(asset.headers['cache-control'], 'public, max-age=31536000, immutable')
assert 'navy' in asset.text
href

'/answerui/523580fc6bd8.css'

A change to the registry gives the next page a new name, and the old name still serves the current build:

In [ ]:
LateCard = hypelt('LateCard', css='late-card { display: block; color: teal; }', layer='components', doc='Defined after the app.')
href2 = re.search(r'href="(/answerui/[^"]+\.css)"', Client(app).get('/').text)[1]
assert href2 != href
assert 'teal' in Client(app).get(href).text

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()